# Preparing the Analysis-Ready NHANES Dataset

This notebook takes the raw merged NHANES dataset produced by `01_data_merging.ipynb` and transforms it into the analysis-ready dataset used for type 2 diabetes prediction research.

Starting from the pooled raw merge across all four NHANES cycles, this notebook applies the following steps in order:

1. **Special missing code replacement** — NHANES survey variables encode refused and don't-know responses as `7`/`9`, `77`/`99`, `777`/`999`, or `7777`/`9999`. These are replaced with `NaN` across all questionnaire and categorical columns.
2. **Harmonization** — Sleep duration and blood pressure are standardised from cycle-specific source variables to consistent column names.
3. **Derived predictors** — PHQ-9 depression score and MET-weighted weekly physical activity are computed from their component variables.
4. **Outcome construction** — The binary `diabetes` label is constructed from clinical and self-reported criteria.
5. **Population criteria** — The sample is restricted to adults aged 20 and older, excluding pregnant participants and those with insufficient outcome information.
6. **Variable selection and renaming** — The final predictor set is selected, outcome-defining variables are excluded to prevent leakage, and all columns are renamed to descriptive English names.

The output is saved to `data/processed/nhanes_analysis_ready.parquet` and serves as the input for `03_eda.ipynb`.

**Note:** This notebook reads `data/raw/nhanes_merged.parquet` produced by `01_data_merging.ipynb`. Run that notebook first if the file does not exist.

In [1]:
import pandas as pd
from pathlib import Path

## Constants and Variable Definitions

This section defines all constants used throughout the preparation pipeline: harmonization source-column groups, the list of survey columns subject to special missing code replacement, the physical activity components used to compute the MET-weighted weekly summary, the final predictor column list, and the column rename map.

In [2]:
# ── Harmonization source columns ─────────────────────────────────────────────

SLEEP_ALTERNATIVES  = ["SLD010H", "SLD012"]

BP_OLD_SYSTOLIC  = ["BPXSY1",  "BPXSY2",  "BPXSY3"]
BP_OLD_DIASTOLIC = ["BPXDI1",  "BPXDI2",  "BPXDI3"]
BP_OSC_SYSTOLIC  = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
BP_OSC_DIASTOLIC = ["BPXODI1", "BPXODI2", "BPXODI3"]

PHQ9_ITEMS = [
    "DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050",
    "DPQ060", "DPQ070", "DPQ080", "DPQ090",
]

# ── Special missing codes ─────────────────────────────────────────────────────

# NHANES encodes Refused (7/77/777/7777) and Don't Know (9/99/999/9999) as
# numeric values rather than NaN. These are replaced before any analysis.
SPECIAL_MISSING_CODES = [7, 9, 77, 99, 777, 999, 7777, 9999]

# Survey and questionnaire columns that use the special missing codes above.
# Continuous laboratory and anthropometric variables are excluded — they use
# a separate missing data system and are unaffected by this replacement.
SURVEY_COLUMNS = [
    # Outcome-defining (needed for outcome construction; excluded from predictors)
    "DIQ010", "DID040", "DIQ050", "DIQ070",
    # Demographics
    "RIAGENDR", "RIDRETH3", "DMDEDUC2", "RIDEXPRG",
    # Lifestyle
    "SMQ020", "ALQ151",
    # Physical activity — binary flags
    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665",
    # Physical activity — raw frequency variables (used for PA summary only)
    "PAQ610", "PAQ625", "PAQ640", "PAQ655", "PAQ670",
    # Sleep
    "SLQ050",
    # PHQ-9 items
    "DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050",
    "DPQ060", "DPQ070", "DPQ080", "DPQ090",
    # Cardiometabolic history
    "BPQ020", "BPQ080",
    # Healthcare access
    "HIQ011", "HIQ210", "HUQ010", "HUQ030", "HUQ051",
    # Food security
    "FSDAD", "FSDHH",
    # Family history
    "MCQ300C",
]

# ── Physical activity components ──────────────────────────────────────────────

# (days_col, minutes_col, intensity_weight)
# Vigorous activities are weighted at 2× moderate-equivalent minutes.
ACTIVITY_COMPONENTS = {
    "vigorous_work_min_week":       ("PAQ610", "PAD615", 2),
    "moderate_work_min_week":       ("PAQ625", "PAD630", 1),
    "transport_walk_bike_min_week": ("PAQ640", "PAD645", 1),
    "vigorous_recreation_min_week": ("PAQ655", "PAD660", 2),
    "moderate_recreation_min_week": ("PAQ670", "PAD675", 1),
}

# ── Final predictor columns ───────────────────────────────────────────────────

PREDICTOR_COLUMNS = [
    # Demographics
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR",
    # Anthropometrics
    "BMXBMI", "BMXWAIST",
    # Blood pressure (harmonized)
    "mean_systolic_bp", "mean_diastolic_bp",
    # Family history
    "MCQ300C",
    # Smoking
    "SMQ020",
    # Alcohol
    "ALQ151",
    # Physical activity — binary flags
    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665",
    # Physical activity — derived summary and sedentary time
    "PAD680",
    "physical_activity_moderate_equivalent_min_week",
    # Sleep
    "sleep_hours", "SLQ050",
    # Mental health
    "phq9_score",
    # Cardiometabolic history
    "BPQ020", "BPQ080",
    # Healthcare access
    "HIQ011", "HIQ210", "HUQ010", "HUQ030", "HUQ051",
    # Food security
    "FSDAD", "FSDHH",
    # Lipids
    "LBXTC", "LBDHDD",
    # Dietary
    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR",
    "DR1TFIBE", "DR1TTFAT", "DR1TCHOL",
]

# ── Column rename map ─────────────────────────────────────────────────────────

COLUMN_RENAME_MAP = {
    "SEQN":     "participant_id",
    "CYCLE":    "cycle",
    "RIDAGEYR": "age",
    "RIAGENDR": "sex",
    "RIDRETH3": "race_ethnicity",
    "DMDEDUC2": "education_level",
    "INDFMPIR": "income_poverty_ratio",
    "BMXBMI":   "bmi",
    "BMXWAIST": "waist_circumference",
    "mean_systolic_bp":  "mean_systolic_bp",
    "mean_diastolic_bp": "mean_diastolic_bp",
    "MCQ300C": "family_history_diabetes",
    "SMQ020":  "ever_smoked_100_cigarettes",
    "ALQ151":  "ever_almost_daily_heavy_drinking",
    "PAQ605":  "vigorous_work_activity",
    "PAQ620":  "moderate_work_activity",
    "PAQ635":  "walk_or_bicycle_transport",
    "PAQ650":  "vigorous_recreational_activity",
    "PAQ665":  "moderate_recreational_activity",
    "PAD680":  "sedentary_minutes_per_day",
    "physical_activity_moderate_equivalent_min_week": "physical_activity_moderate_equivalent_min_week",
    "sleep_hours": "sleep_hours",
    "SLQ050":      "trouble_sleeping_reported_to_doctor",
    "phq9_score":  "phq9_score",
    "BPQ020": "ever_told_hypertension",
    "BPQ080": "ever_told_high_cholesterol",
    "HIQ011": "has_health_insurance",
    "HIQ210": "insurance_gap_past_12_months",
    "HUQ010": "self_rated_health",
    "HUQ030": "usual_place_for_healthcare",
    "HUQ051": "healthcare_visits_past_12_months",
    "FSDAD":  "adult_food_security_category",
    "FSDHH":  "household_food_security_category",
    "LBXTC":    "total_cholesterol_mg_dl",
    "LBDHDD":   "hdl_cholesterol_mg_dl",
    "DR1TKCAL": "energy_kcal",
    "DR1TPROT": "protein_g",
    "DR1TCARB": "carbohydrate_g",
    "DR1TSUGR": "total_sugar_g",
    "DR1TFIBE": "fiber_g",
    "DR1TTFAT": "total_fat_g",
    "DR1TCHOL": "cholesterol_mg",
}

## Loading the Raw Merged Dataset

The raw merged dataset produced by `01_data_merging.ipynb` is loaded from `data/raw/nhanes_merged.parquet`. This file contains one row per participant across all four NHANES cycles, with original NHANES variable names and no transformations applied.

All harmonization source columns that may be absent from the merged dataset (because a given cycle lacks them) are filled with `NA` here so that the harmonization steps below run without errors regardless of which cycles are present.

In [4]:
input_path = Path("../data/raw/nhanes_diabetes_raw.parquet")
df = pd.read_parquet(input_path)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print(df["CYCLE"].value_counts())

# Ensure all harmonization source columns exist.
# Cycles that lack a column (e.g. the 2017-2020 cycle lacks standard BP columns)
# will have NA values for those columns, handled downstream by the fillna logic.
all_source_cols = (
    SLEEP_ALTERNATIVES
    + BP_OLD_SYSTOLIC + BP_OLD_DIASTOLIC
    + BP_OSC_SYSTOLIC + BP_OSC_DIASTOLIC
)
for col in all_source_cols:
    if col not in df.columns:
        df[col] = pd.NA

Loaded: 45,462 rows x 769 columns

CYCLE
2017-March 2020    15560
2013-2014          10175
2015-2016           9971
2011-2012           9756
Name: count, dtype: int64


## Replacing Special Missing Codes

NHANES survey variables encode two types of non-response as numeric codes rather than `NaN`:

| Code | Meaning |
|------|---------|
| 7 / 77 / 777 / 7777 | Refused |
| 9 / 99 / 999 / 9999 | Don't know |

These codes appear across all questionnaire and categorical variables and must be replaced with `NaN` before any analysis. Continuous variables such as laboratory measurements, anthropometrics, and dietary totals use a separate missing data system and are not affected by this step.

The replacement covers all columns listed in `SURVEY_COLUMNS`, including the PHQ-9 items, so no separate per-item replacement is needed later.

In [5]:
survey_cols_present = [col for col in SURVEY_COLUMNS if col in df.columns]
df[survey_cols_present] = df[survey_cols_present].replace(SPECIAL_MISSING_CODES, pd.NA)

print(f"Special missing codes replaced in {len(survey_cols_present)} of {len(SURVEY_COLUMNS)} survey columns.")
not_found = [c for c in SURVEY_COLUMNS if c not in df.columns]
if not_found:
    print(f"Columns not found in dataset: {not_found}")

Special missing codes replaced in 40 of 40 survey columns.


## Harmonizing Cycle-Inconsistent Variables

Two variable groups use different names across NHANES cycles and require harmonization before the pooled dataset can be used for analysis.

| Group | Source variables | Harmonized column | Notes |
|---|---|---|---|
| Sleep duration | `SLD010H` (2011–2016), `SLD012` (2017–2020) | `sleep_hours` | Weekday sleep in hours |
| Systolic BP | `BPXSY1–3` (2011–2016), `BPXOSY1–3` (2017–2020) | `mean_systolic_bp` | Mean of available readings |
| Diastolic BP | `BPXDI1–3` (2011–2016), `BPXODI1–3` (2017–2020) | `mean_diastolic_bp` | Mean of available readings |

For blood pressure, the mean across available readings is computed for the standard format first, then filled from the oscillometric mean where the standard reading is absent. Because the two formats do not overlap across cycles, this is equivalent to using whichever format a given participant's cycle provides.

In [6]:
# Sleep duration
available_sleep = [c for c in SLEEP_ALTERNATIVES if c in df.columns]
df["sleep_hours"] = df[available_sleep].bfill(axis=1).iloc[:, 0]

print(f"sleep_hours: {df['sleep_hours'].notna().sum():,} non-null")
print(f"  Source columns used: {available_sleep}")

sleep_hours: 29,026 non-null
  Source columns used: ['SLD010H', 'SLD012']


In [7]:
# Blood pressure
df["mean_systolic_bp"] = (
    df[BP_OLD_SYSTOLIC].mean(axis=1)
    .fillna(df[BP_OSC_SYSTOLIC].mean(axis=1))
)
df["mean_diastolic_bp"] = (
    df[BP_OLD_DIASTOLIC].mean(axis=1)
    .fillna(df[BP_OSC_DIASTOLIC].mean(axis=1))
)

print(f"mean_systolic_bp:  {df['mean_systolic_bp'].notna().sum():,} non-null")
print(f"mean_diastolic_bp: {df['mean_diastolic_bp'].notna().sum():,} non-null")

mean_systolic_bp:  32,295 non-null
mean_diastolic_bp: 32,295 non-null


## Constructing Derived Predictors

Two predictors are computed from multiple source variables rather than used directly.

**PHQ-9 depression score** (`phq9_score`) is constructed by summing the nine PHQ-9 item scores (`DPQ010`–`DPQ090`). Special missing codes for these items were already replaced with `NaN` in the step above. `min_count=1` ensures that participants with at least one valid item receive a partial score rather than `NaN`.

**MET-weighted weekly physical activity** (`physical_activity_moderate_equivalent_min_week`) is calculated from raw activity frequency (days per week) and duration (minutes per session) variables across five activity domains. Vigorous activities are weighted at 2x moderate-equivalent minutes; all other activity types at 1x. The raw component variables are not retained as standalone predictors in the final dataset.

In [8]:
# PHQ-9 score
df["phq9_score"] = df[PHQ9_ITEMS].sum(axis=1, min_count=1)

print(f"phq9_score: {df['phq9_score'].notna().sum():,} non-null")
print(df["phq9_score"].describe().round(1))

phq9_score: 23,816 non-null
count     23816.0
unique       33.0
top           0.0
freq       7784.0
Name: phq9_score, dtype: float64


In [9]:
# MET-weighted weekly physical activity
activity_summary_cols = []
for new_col, (days_col, minutes_col, weight) in ACTIVITY_COMPONENTS.items():
    days    = pd.to_numeric(df.get(days_col,    pd.Series(pd.NA, index=df.index)), errors="coerce")
    minutes = pd.to_numeric(df.get(minutes_col, pd.Series(pd.NA, index=df.index)), errors="coerce")
    df[new_col] = days * minutes * weight
    activity_summary_cols.append(new_col)

df["physical_activity_moderate_equivalent_min_week"] = (
    df[activity_summary_cols].sum(axis=1, min_count=1)
)

print(
    f"physical_activity_moderate_equivalent_min_week: "
    f"{df['physical_activity_moderate_equivalent_min_week'].notna().sum():,} non-null"
)

physical_activity_moderate_equivalent_min_week: 21,198 non-null


## Constructing the Diabetes Outcome Variable

The binary `diabetes` outcome is constructed using a composite definition: a participant is classified as having diabetes if they meet at least one of the following criteria:

| Criterion | Variable | Threshold |
|---|---|---|
| Self-reported diagnosis | `DIQ010` | == 1 |
| HbA1c | `LBXGH` | >= 6.5% |
| Fasting plasma glucose | `LBXGLU` | >= 126 mg/dL |
| Current insulin use | `DIQ050` | == 1 |
| Diabetes medication use | `DIQ070` | == 1 |

Participants for whom none of these variables are available are assigned `NaN` rather than being incorrectly labelled as non-diabetic. These participants are excluded in the population criteria step below.

None of the outcome-defining variables are included in the predictor set to prevent leakage.

In [10]:
self_report     = df["DIQ010"] == 1
hba1c           = df["LBXGH"]  >= 6.5
fasting_glucose = df["LBXGLU"] >= 126
insulin_use     = df["DIQ050"] == 1
diabetes_med    = df["DIQ070"] == 1

has_outcome_info = (
    df["DIQ010"].notna()
    | df["LBXGH"].notna()
    | df["LBXGLU"].notna()
    | df["DIQ050"].notna()
    | df["DIQ070"].notna()
)

df["diabetes"] = (
    self_report | hba1c | fasting_glucose | insulin_use | diabetes_med
).astype("Int64")

df.loc[~has_outcome_info, "diabetes"] = pd.NA

print(df["diabetes"].value_counts(dropna=False))
print()
print(df["diabetes"].value_counts(normalize=True, dropna=True).round(3))

diabetes
0       38688
1        5001
<NA>     1773
Name: count, dtype: Int64

diabetes
0    0.886
1    0.114
Name: proportion, dtype: Float64


## Applying Population Criteria

The study population is restricted using the following inclusion and exclusion criteria:

| Criterion | Rationale |
|---|---|
| Age >= 20 years | Type 2 diabetes prediction is targeted at adults |
| At least one outcome-defining variable available | Participants without any outcome information cannot be reliably labelled |
| Not currently pregnant (`RIDEXPRG != 1`) | Pregnancy affects metabolic markers used in the outcome definition |

Row counts before and after filtering are printed, along with the rechecked class distribution.

In [11]:
n_before = len(df)

df = df[df["diabetes"].notna()]
df = df[df["RIDAGEYR"] >= 20]

if "RIDEXPRG" in df.columns:
    df = df[df["RIDEXPRG"] != 1]

n_after = len(df)

print(f"Population filtering: {n_before:,} -> {n_after:,} rows ({n_before - n_after:,} excluded)")
print()
print(df["diabetes"].value_counts(dropna=False))
print()
print(df["diabetes"].value_counts(normalize=True, dropna=True).round(3))

Population filtering: 45,462 -> 26,000 rows (19,462 excluded)

diabetes
0    21091
1     4909
Name: count, dtype: Int64

diabetes
0    0.811
1    0.189
Name: proportion, dtype: Float64


## Selecting and Renaming Analysis Variables

The final analysis dataset is assembled by extracting the participant identifier, survey cycle, diabetes outcome, and the predictor columns listed in `PREDICTOR_COLUMNS`.

Any predictor column absent from the dataset (due to a merge issue or complete missingness across cycles) is filled with `NA` so that the output schema remains consistent. Downstream notebooks can rely on all expected columns being present.

All NHANES source codes are then renamed to descriptive English names using `COLUMN_RENAME_MAP`.

In [12]:
# Fill any predictor columns absent from the dataset with NA
for col in PREDICTOR_COLUMNS:
    if col not in df.columns:
        df[col] = pd.NA

analysis_df = df[["SEQN", "CYCLE", "diabetes"] + PREDICTOR_COLUMNS].copy()
analysis_df = analysis_df.rename(columns=COLUMN_RENAME_MAP)

print(f"Final dataset: {analysis_df.shape[0]:,} rows x {analysis_df.shape[1]} columns")
analysis_df.head()

Final dataset: 26,000 rows x 43 columns


,participant_id,cycle,diabetes,age,sex,race_ethnicity,education_level,income_poverty_ratio,bmi,waist_circumference,...,household_food_security_category,total_cholesterol_mg_dl,hdl_cholesterol_mg_dl,energy_kcal,protein_g,carbohydrate_g,total_sugar_g,fiber_g,total_fat_g,cholesterol_mg
0,62161.0,2011-2012,0,22.0,1.0,3.0,3.0,3.15,23.3,81.0,...,1.0,168.0,41.0,2969.0,104.68,359.59,109.09,18.6,123.81,328.0
3,62164.0,2011-2012,0,44.0,2.0,3.0,4.0,1.67,23.2,80.1,...,1.0,190.0,28.0,1115.0,73.13,91.67,32.29,9.5,51.54,207.0
8,62169.0,2011-2012,0,21.0,1.0,6.0,3.0,0.33,20.1,69.6,...,3.0,132.0,43.0,1831.0,77.46,297.51,78.19,4.3,34.61,205.0
11,62172.0,2011-2012,0,43.0,2.0,4.0,3.0,2.02,33.3,120.4,...,1.0,169.0,73.0,1845.0,57.43,192.82,58.56,2.8,42.02,160.0
13,62174.0,2011-2012,0,80.0,1.0,3.0,5.0,4.30,33.9,116.5,...,1.0,203.0,54.0,1178.0,39.35,146.72,84.63,12.0,49.75,468.0


## Saving the Analysis-Ready Dataset

The prepared dataset is saved to `data/processed/nhanes_analysis_ready.parquet`.

In [13]:
output_path = Path("../data/processed/nhanes_analysis_ready.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

analysis_df.to_parquet(output_path, index=False)

print(f"Saved to:  {output_path}")
print(f"Rows:      {analysis_df.shape[0]:,}")
print(f"Columns:   {analysis_df.shape[1]}")

Saved to:  ../data/processed/nhanes_analysis_ready.parquet
Rows:      26,000
Columns:   43
